# Flow Matching Testing
Comprehensive testing for flow matching genre transformation

In [ ]:
import torch
from models.flow import FlowMatching
from models.dit import DiT


film_conditioner.py STARTED


In [5]:

# Fake data
B = 2
C = 1
H = 80
W = 256

x0 = torch.randn(B, C, H, W)  # non-rock
x1 = torch.randn(B, C, H, W)  # rock
genre_ids = torch.tensor([1, 1])  # target = rock

dit = DiT()
flow = FlowMatching(dit)

loss = flow.compute_loss(x0, x1, genre_ids)
print("Loss:", loss.item())


Loss: 2.341831684112549


In [7]:
with torch.no_grad():
    t = torch.rand(B)
    xt = (1 - t.view(B,1,1,1)) * x0 + t.view(B,1,1,1) * x1
    v_true = x1 - x0
    v_pred = dit(xt, t, genre_ids)

print("True velocity norm:", v_true.norm().item())
print("Pred velocity norm:", v_pred.norm().item())


True velocity norm: 286.49859619140625
Pred velocity norm: 116.85894012451172


In [14]:
optimizer = torch.optim.Adam(dit.parameters(), lr=1e-4)

for step in range(200):
    optimizer.zero_grad()
    loss = flow.compute_loss(x0, x1, genre_ids)
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        
        print(f"{step:4d} → {loss.item()}")


   0 → 1.9766470193862915
  10 → 1.8476394414901733
  20 → 1.8762695789337158
  30 → 1.691589117050171
  40 → 1.7182104587554932
  50 → 1.8100610971450806
  60 → 1.759728193283081
  70 → 1.4647516012191772
  80 → 1.6594884395599365
  90 → 1.6924397945404053
 100 → 1.6776759624481201
 110 → 1.52814519405365
 120 → 1.3715074062347412
 130 → 1.4905577898025513
 140 → 1.2056186199188232
 150 → 1.3099126815795898
 160 → 1.2230743169784546
 170 → 1.087341070175171
 180 → 1.0562349557876587
 190 → 0.9476248025894165


In [13]:
with torch.no_grad():
    x_gen = flow.sample_euler(x0, genre_ids=1, num_steps=50)

dist_start = torch.mean(torch.abs(x0 - x1))
dist_end = torch.mean(torch.abs(x_gen - x1))

print("Distance before:", dist_start.item())
print("Distance after :", dist_end.item())


Distance before: 1.1282652616500854
Distance after : 1.0372235774993896


In [15]:
with torch.no_grad():
    x_euler = flow.sample_euler(x0, 1, num_steps=30)
    x_heun = flow.sample_heun(x0, 1, num_steps=30)

print("Euler vs Heun difference:",
      torch.mean(torch.abs(x_euler - x_heun)).item())


Euler vs Heun difference: 0.0378994345664978
